In [2]:
import numpy as np
import xarray as xr
import netCDF4 as nc
from skimage.transform import resize
import os

## Process constant

In [4]:
# ── Config ────────────────────────────────────────────────────────────────────
INPUT_NC   = "/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/constant.nc"
OUTPUT_NPZ = "constant.npz"

OROG_VAR = "orog"   # change if your variable name differs
LSM_VAR  = "lsm"   # change if your variable name differs

TARGET_SIZES = {
    "5.5km":  (1068, 1068),
    "11km":   (534,  534),
    "22km":   (267,  267),
    "44km":   (133,  133),
}
# ─────────────────────────────────────────────────────────────────────────────

def resize_field(field, target_shape, anti_alias=True):
    """Bilinear resize (order=1). Returns float32."""
    return resize(
        field,
        target_shape,
        order=1,                  # bilinear
        mode="edge",              # pad with edge values
        anti_aliasing=anti_alias,
        preserve_range=True,
    ).astype(np.float32)


def main():
    # 1. Load
    ds   = nc.Dataset(INPUT_NC, "r")
    orog = np.array(ds[OROG_VAR][:], dtype=np.float32).squeeze()  # (1069, 1069)
    lsm  = np.array(ds[LSM_VAR][:],  dtype=np.float32).squeeze()  # (1069, 1069)
    ds.close()

    print(f"Loaded  orog {orog.shape}  lsm {lsm.shape}")

    # 2. Resize
    out = {}
    for tag, shape in TARGET_SIZES.items():
        orog_r = resize_field(orog, shape, anti_alias=True)
        lsm_r  = resize_field(lsm,  shape, anti_alias=True)

        # Threshold LSM back to binary
        lsm_r  = (lsm_r > 0.5).astype(np.float32)

        out["orography"] = orog_r
        out["lsm"]  = lsm_r

        print(f"  [{tag}] orog {orog_r.shape}  lsm {lsm_r.shape}"
              f"  orog range [{orog_r.min():.1f}, {orog_r.max():.1f}]")

        # 4. Save
        save_path = os.path.join("/Users/thainamhoang/Desktop/ph-reg/data/cerra", tag, OUTPUT_NPZ)
        np.savez_compressed(save_path, **out)
        print(f"\nSaved → {save_path}")
        print("Keys:", list(out.keys()))
main()

Loaded  orog (1069, 1069)  lsm (1069, 1069)
  [5.5km] orog (1068, 1068)  lsm (1068, 1068)  orog range [-407.9, 3937.5]

Saved → /Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/constant.npz
Keys: ['orography', 'lsm']
  [11km] orog (534, 534)  lsm (534, 534)  orog range [-366.2, 3637.2]

Saved → /Users/thainamhoang/Desktop/ph-reg/data/cerra/11km/constant.npz
Keys: ['orography', 'lsm']
  [22km] orog (267, 267)  lsm (267, 267)  orog range [-177.9, 3216.1]

Saved → /Users/thainamhoang/Desktop/ph-reg/data/cerra/22km/constant.npz
Keys: ['orography', 'lsm']
  [44km] orog (133, 133)  lsm (133, 133)  orog range [-75.1, 3201.2]

Saved → /Users/thainamhoang/Desktop/ph-reg/data/cerra/44km/constant.npz
Keys: ['orography', 'lsm']


In [3]:
ds = xr.open_dataset("/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/2m_temperature/2010.nc")
print(ds)
print(ds.data_vars)
print(ds.coords)

<xarray.Dataset> Size: 13GB
Dimensions:     (valid_time: 2920, y: 1069, x: 1069)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 23kB 2010-01-01 ... 2010-12-31T21...
    expver      (valid_time) <U4 47kB ...
    latitude    (y, x) float64 9MB ...
    longitude   (y, x) float64 9MB ...
Dimensions without coordinates: y, x
Data variables:
    t2m         (valid_time, y, x) float32 13GB ...
Attributes:
    GRIB_centre:             eswi
    GRIB_centreDescription:  Norrkoping
    GRIB_subCentre:          255
    Conventions:             CF-1.7
    institution:             Norrkoping
    history:                 2026-03-19T11:17 GRIB to CDM+CF via cfgrib-0.9.1...
Data variables:
    t2m      (valid_time, y, x) float32 13GB ...
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 23kB 2010-01-01 ... 2010-12-31T21...
    expver      (valid_time) <U4 47kB ...
    latitude    (y, x) float64 9MB ...
    longitude   (y, x) float64 9MB ...


In [4]:
ds = xr.open_dataset("/Users/thainamhoang/Desktop/ph-reg/data/5.625deg/2m_temperature/2m_temperature_5.625deg/2m_temperature_1979_5.625deg.nc")
print(ds)
print(ds.data_vars)
print(ds.coords)

<xarray.Dataset> Size: 72MB
Dimensions:  (time: 8760, lat: 32, lon: 64)
Coordinates:
  * time     (time) datetime64[ns] 70kB 1979-01-01 ... 1979-12-31T23:00:00
  * lat      (lat) float64 256B -87.19 -81.56 -75.94 ... 75.94 81.56 87.19
  * lon      (lon) float64 512B 0.0 5.625 11.25 16.88 ... 343.1 348.8 354.4
Data variables:
    t2m      (time, lat, lon) float32 72MB ...
Attributes:
    Conventions:  CF-1.6
    history:      2019-11-07 07:51:20 GMT by grib_to_netcdf-2.14.0: /opt/ecmw...
Data variables:
    t2m      (time, lat, lon) float32 72MB ...
Coordinates:
  * time     (time) datetime64[ns] 70kB 1979-01-01 ... 1979-12-31T23:00:00
  * lat      (lat) float64 256B -87.19 -81.56 -75.94 ... 75.94 81.56 87.19
  * lon      (lon) float64 512B 0.0 5.625 11.25 16.88 ... 343.1 348.8 354.4


In [6]:
_mock = np.load("/Users/thainamhoang/Downloads/1979_0.npz")
_mock["2m_temperature"]

array([[[[249.5468 , 249.2551 , 248.99257, ..., 250.56598, 250.0975 ,
          249.76407],
         [252.58041, 250.45457, 249.3099 , ..., 252.97847, 252.4582 ,
          253.20105],
         [245.92776, 243.87862, 243.09381, ..., 261.1736 , 251.56703,
          249.9961 ],
         ...,
         [262.2742 , 262.4549 , 259.07263, ..., 250.16113, 249.09265,
          251.27164],
         [248.38705, 248.66263, 248.64355, ..., 244.76285, 248.76778,
          248.81682],
         [252.40489, 252.56508, 252.6287 , ..., 251.8821 , 252.03676,
          252.19366]]],


       [[[249.4784 , 249.18721, 248.93123, ..., 250.4415 , 249.98636,
          249.67227],
         [252.37473, 250.28307, 249.15048, ..., 252.77026, 252.2309 ,
          252.9604 ],
         [245.56265, 243.69556, 243.08879, ..., 260.422  , 251.31859,
          249.26985],
         ...,
         [262.13943, 262.44604, 258.7191 , ..., 249.8634 , 248.72401,
          250.93268],
         [248.93022, 249.24124, 249.41151, ..., 

In [6]:
"""
CERRA preprocessing script — ERA5-compatible output format.

Memory-safe: processes and saves data chunk-by-chunk, never materializing
a full year in memory at once.

Target grid sizes:
    1068 x 1068  (5.5 km)
    534  x 534   (11 km)
    267  x 267   (22 km)
    133  x 133   (44 km)

Timestep handling:
    Normal year : 365 * 8 = 2920  ->  2912  (182 * 16)
    Leap year   : 366 * 8 = 2928  ->  2928  (183 * 16)

Split: train 2010-2019 | val 2020 | test 2021
"""

import calendar
import glob
import os

import dask.array as da
import numpy as np
import xarray as xr
from dask.distributed import Client, LocalCluster
from skimage.transform import resize
from tqdm import tqdm

# ── Variable mappings ─────────────────────────────────────────────────────────
NAME_TO_VAR = {
    "2m_temperature":          "t2m",
    "10m_u_component_of_wind": "u10",
    "10m_v_component_of_wind": "v10",
    "2m_relative_humidity":    "r2",
    "surface_pressure":        "sp",
    "land_sea_mask":           "lsm",
    "orography":               "orog",
}

VAR_TO_NAME        = {v: k for k, v in NAME_TO_VAR.items()}
NATIVE_SIZE        = 1069
TIMESTAMPS_PER_DAY = 8


# ── Helpers ───────────────────────────────────────────────────────────────────
def steps_per_year(year: int, round_to: int = 16) -> int:
    days  = 366 if calendar.isleap(year) else 365
    total = days * TIMESTAMPS_PER_DAY
    return (total // round_to) * round_to


def _resize_block(block: np.ndarray, target_size: int, block_info=None) -> np.ndarray:
    """
    Resize block (T, 1, H, W) -> (T, 1, target_size, target_size).
    Pure numpy — called per Dask block via map_blocks.
    """
    T   = block.shape[0]
    out = np.empty((T, 1, target_size, target_size), dtype=np.float32)
    for i in range(T):
        out[i, 0] = resize(
            block[i, 0],
            (target_size, target_size),
            order=1,
            mode="edge",
            anti_aliasing=True,
            preserve_range=True,
        ).astype(np.float32)
    return out


def make_dask_array(ds: xr.Dataset, code: str, n_steps: int, chunk_size: int) -> da.Array:
    """
    Build a lazy dask array (n_steps, 1, 1069, 1069) from xarray dataset.
    """
    data = ds[code].data                        # dask array (valid_time, y, x)
    data = data[:n_steps]                       # truncate
    data = data[:, np.newaxis, :, :]            # add level dim
    data = data.rechunk({0: chunk_size, 1: 1,
                         2: NATIVE_SIZE, 3: NATIVE_SIZE})
    return data


def resize_dask(data: da.Array, target_size: int) -> da.Array:
    """Register map_blocks resize — still lazy."""
    return data.map_blocks(
        _resize_block,
        target_size=target_size,
        dtype=np.float32,
        chunks=(
            data.chunks[0],
            (1,) * len(data.chunks[1]),
            (target_size,) * len(data.chunks[2]),
            (target_size,) * len(data.chunks[3]),
        ),
    )


def extract_1d_coords(ds: xr.Dataset, target_size: int):
    lat2d      = ds["latitude"].to_numpy()
    lon2d      = ds["longitude"].to_numpy()
    native_idx = np.linspace(0, NATIVE_SIZE - 1, NATIVE_SIZE)
    target_idx = np.linspace(0, NATIVE_SIZE - 1, target_size)
    lat = np.interp(target_idx, native_idx, lat2d[:, 0]).astype(np.float64)
    lon = np.interp(target_idx, native_idx, lon2d[0, :]).astype(np.float64)
    return lat, lon


# ── Core function ─────────────────────────────────────────────────────────────
def nc2np(
    path: str,
    variables: list,
    years,
    save_dir: str,
    partition: str,
    num_shards_per_year: int,
    grid_size: int,
    chunk_size: int,
):
    """
    Process CERRA NetCDF files chunk-by-chunk to avoid OOM.

    Strategy:
    - Build lazy Dask graph for full year
    - Iterate over shards, computing only `steps_per_shard` frames at a time
    - Accumulate stats incrementally (Welford online algorithm for std)
    """
    out_dir = os.path.join(save_dir, partition)
    os.makedirs(out_dir, exist_ok=True)

    if partition == "train":
        normalize_mean = {}
        normalize_std  = {}
    climatology = {}

    for year in tqdm(years, desc=f"{partition}"):
        n_steps         = steps_per_year(year)
        steps_per_shard = n_steps // num_shards_per_year

        assert n_steps % num_shards_per_year == 0, (
            f"n_steps={n_steps} not divisible by num_shards_per_year={num_shards_per_year}"
        )

        # Per-year accumulators for stats (train only)
        if partition == "train":
            year_sum    = {}
            year_sum_sq = {}
            year_count  = {}

        clim_acc = {}

        for var in variables:
            ps = glob.glob(os.path.join(path, var, f"*{year}*.nc"))
            if not ps:
                raise FileNotFoundError(
                    f"No files for '{var}' year {year} under {os.path.join(path, var)}/"
                )

            ds   = xr.open_mfdataset(ps, combine="by_coords",
                                     chunks={"valid_time": chunk_size})
            code = NAME_TO_VAR[var]

            assert ds[code].shape[1] == NATIVE_SIZE and ds[code].shape[2] == NATIVE_SIZE, (
                f"Expected native ({NATIVE_SIZE},{NATIVE_SIZE}), got {ds[code].shape[1:]}"
            )

            # Build full lazy graph for the year (no memory used yet)
            data_lazy = make_dask_array(ds, code, n_steps, chunk_size)
            data_lazy = resize_dask(data_lazy, grid_size)
            # shape: (n_steps, 1, grid_size, grid_size) — still lazy

            if partition == "train":
                year_sum[var]    = None
                year_sum_sq[var] = None
                year_count[var]  = 0
            clim_acc[var] = None

            # ── Shard loop: compute one shard at a time ───────────────────────
            for shard_id in range(num_shards_per_year):
                s = shard_id * steps_per_shard
                e = s + steps_per_shard

                # Slice the lazy array — only this shard is computed
                shard_lazy = data_lazy[s:e]   # (steps_per_shard, 1, H, W)

                print(f"  [{partition}] {var} {year} shard {shard_id+1}/"
                      f"{num_shards_per_year} "
                      f"steps [{s}:{e}] -> computing...")

                shard_np = shard_lazy.compute().astype(np.float32)
                # shape: (steps_per_shard, 1, grid_size, grid_size)

                # Save shard
                np.savez(
                    os.path.join(out_dir, f"{year}_{shard_id}.npz"),
                    **{var: shard_np},
                )

                # Accumulate stats over shards
                if partition == "train":
                    s_sum    = shard_np.sum(axis=(0, 2, 3))    # (1,)
                    s_sum_sq = (shard_np ** 2).sum(axis=(0, 2, 3))
                    s_count  = shard_np.shape[0] * shard_np.shape[2] * shard_np.shape[3]

                    if year_sum[var] is None:
                        year_sum[var]    = s_sum
                        year_sum_sq[var] = s_sum_sq
                        year_count[var]  = s_count
                    else:
                        year_sum[var]    += s_sum
                        year_sum_sq[var] += s_sum_sq
                        year_count[var]  += s_count

                # Accumulate climatology (mean over time axis per shard)
                shard_clim = shard_np.mean(axis=0)   # (1, H, W)
                if clim_acc[var] is None:
                    clim_acc[var] = shard_clim
                else:
                    clim_acc[var] += shard_clim

            # Average climatology across shards
            clim_yearly = clim_acc[var] / num_shards_per_year
            if var not in climatology:
                climatology[var] = [clim_yearly]
            else:
                climatology[var].append(clim_yearly)

            # Yearly mean/std from accumulators
            if partition == "train":
                n    = year_count[var]
                mean = year_sum[var] / n
                std  = np.sqrt(year_sum_sq[var] / n - mean ** 2)

                if var not in normalize_mean:
                    normalize_mean[var] = [mean]
                    normalize_std[var]  = [std]
                else:
                    normalize_mean[var].append(mean)
                    normalize_std[var].append(std)

    # ── Normalisation stats (train only) ─────────────────────────────────────
    if partition == "train":
        for var in normalize_mean:
            normalize_mean[var] = np.stack(normalize_mean[var], axis=0)
            normalize_std[var]  = np.stack(normalize_std[var],  axis=0)

        for var in normalize_mean:
            mean     = normalize_mean[var]
            std      = normalize_std[var]
            variance = (
                (std ** 2).mean(axis=0)
                + (mean ** 2).mean(axis=0)
                - mean.mean(axis=0) ** 2
            )
            normalize_mean[var] = mean.mean(axis=0)
            normalize_std[var]  = np.sqrt(variance)

        np.savez(os.path.join(save_dir, "normalize_mean.npz"), **normalize_mean)
        np.savez(os.path.join(save_dir, "normalize_std.npz"),  **normalize_std)

    # ── Climatology ───────────────────────────────────────────────────────────
    for var in climatology:
        climatology[var] = np.mean(np.stack(climatology[var], axis=0), axis=0)
    np.savez(os.path.join(out_dir, "climatology.npz"), **climatology)


# ── Entry point ───────────────────────────────────────────────────────────────
def convert_nc2npz(
    root_dir: str,
    save_dir: str,
    variables: list,
    start_train_year: int,
    start_val_year: int,
    start_test_year: int,
    end_year: int,
    num_shards: int,
    grid_size: int,
    chunk_size: int = 32,
    n_workers: int = None,
    threads_per_worker: int = 2,
    memory_limit: str = "4GB",
):
    """
    Full pipeline with a Dask LocalCluster.

    Memory tuning:
        With 24 GB RAM, a safe split is e.g.:
            n_workers=4, threads_per_worker=2, memory_limit="5GB"
        Each shard at 534^2 float32 is ~(728 * 1 * 534 * 534 * 4) ≈ 0.8 GB
        so a worker can comfortably hold one shard at a time.

    Args:
        chunk_size         : Frames per Dask block. 32 is a safe default.
        n_workers          : Dask worker processes. None = os.cpu_count().
        threads_per_worker : Threads per worker process.
        memory_limit       : RAM cap per worker (e.g. "5GB", "4GB").
    """
    assert start_val_year  > start_train_year
    assert start_test_year > start_val_year
    assert end_year        > start_test_year

    train_years = range(start_train_year, start_val_year)
    val_years   = range(start_val_year,   start_test_year)
    test_years  = range(start_test_year,  end_year)

    os.makedirs(save_dir, exist_ok=True)

    cluster = LocalCluster(
        n_workers=n_workers,
        threads_per_worker=threads_per_worker,
        memory_limit=memory_limit,
    )
    client = Client(cluster)
    print(f"Dask dashboard: {client.dashboard_link}")

    try:
        for partition, years in [
            ("train", train_years),
            ("val",   val_years),
            ("test",  test_years),
        ]:
            nc2np(root_dir, variables, years, save_dir, partition,
                  num_shards, grid_size, chunk_size)

        ps       = glob.glob(os.path.join(root_dir, variables[0], f"*{train_years[0]}*.nc"))
        ds       = xr.open_mfdataset(ps[0], parallel=True)
        lat, lon = extract_1d_coords(ds, grid_size)
        np.save(os.path.join(save_dir, "lat.npy"), lat)
        np.save(os.path.join(save_dir, "lon.npy"), lon)
        print(f"Saved lat {lat.shape} lon {lon.shape} -> {save_dir}/")

    finally:
        client.close()
        cluster.close()

In [5]:
# ps       = glob.glob(os.path.join(root_dir, variables[0], f"*{train_years[0]}*.nc"))
ds       = xr.open_mfdataset("/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/raw_t2m/2m_temperature/2010.nc", parallel=True)
lat, lon = extract_1d_coords(ds, 1068)
np.save(os.path.join("/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/2m_temperature", "lat.npy"), lat)
np.save(os.path.join("/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/2m_temperature", "lon.npy"), lon)
print(f"Saved lat {lat.shape} lon {lon.shape} -> {"/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/2m_temperature"}/")

Saved lat (1068,) lon (1068,) -> /Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/2m_temperature/


In [7]:
convert_nc2npz(
        root_dir         = "/Users/thainamhoang/Desktop/ph-reg/data/cerra/5.5km/raw_t2m",
        save_dir         = "/Users/thainamhoang/Desktop/ph-reg/data/cerra/44km/2m_temperature",
        variables        = ["2m_temperature"],
        start_train_year = 2010,
        start_val_year   = 2020,
        start_test_year  = 2021,
        end_year         = 2022,
        num_shards       = 16,
        grid_size        = 133,   # 1068 | 534 | 267 | 133
        chunk_size         = 32,    # frames per dask block
        n_workers          = 4,     # tune to your machine
        threads_per_worker = 2,
        memory_limit       = "5GB", # 4 workers * 5GB = 20GB < 24GB
    )

Dask dashboard: http://127.0.0.1:8787/status


train:   0%|          | 0/10 [00:00<?, ?it/s]

  [train] 2m_temperature 2010 shard 1/16 steps [0:182] -> computing...


/var/folders/44/hp8ptt8n3j16gtl4vny36krh0000gn/T/ipykernel_25415/2925846405.py:160: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 32. This could degrade performance. Instead, consider rechunking after loading.
  ds   = xr.open_mfdataset(ps, combine="by_coords",


  [train] 2m_temperature 2010 shard 2/16 steps [182:364] -> computing...
  [train] 2m_temperature 2010 shard 3/16 steps [364:546] -> computing...
  [train] 2m_temperature 2010 shard 4/16 steps [546:728] -> computing...
  [train] 2m_temperature 2010 shard 5/16 steps [728:910] -> computing...
  [train] 2m_temperature 2010 shard 6/16 steps [910:1092] -> computing...
  [train] 2m_temperature 2010 shard 7/16 steps [1092:1274] -> computing...
  [train] 2m_temperature 2010 shard 8/16 steps [1274:1456] -> computing...
  [train] 2m_temperature 2010 shard 9/16 steps [1456:1638] -> computing...
  [train] 2m_temperature 2010 shard 10/16 steps [1638:1820] -> computing...
  [train] 2m_temperature 2010 shard 11/16 steps [1820:2002] -> computing...
  [train] 2m_temperature 2010 shard 12/16 steps [2002:2184] -> computing...
  [train] 2m_temperature 2010 shard 13/16 steps [2184:2366] -> computing...
  [train] 2m_temperature 2010 shard 14/16 steps [2366:2548] -> computing...
  [train] 2m_temperature 2010

...

  [val] 2m_temperature 2020 shard 1/16 steps [0:183] -> computing...
  [val] 2m_temperature 2020 shard 2/16 steps [183:366] -> computing...
  [val] 2m_temperature 2020 shard 3/16 steps [366:549] -> computing...
  [val] 2m_temperature 2020 shard 4/16 steps [549:732] -> computing...
  [val] 2m_temperature 2020 shard 5/16 steps [732:915] -> computing...
  [val] 2m_temperature 2020 shard 6/16 steps [915:1098] -> computing...
  [val] 2m_temperature 2020 shard 7/16 steps [1098:1281] -> computing...
  [val] 2m_temperature 2020 shard 8/16 steps [1281:1464] -> computing...
  [val] 2m_temperature 2020 shard 9/16 steps [1464:1647] -> computing...
  [val] 2m_temperature 2020 shard 10/16 steps [1647:1830] -> computing...
  [val] 2m_temperature 2020 shard 11/16 steps [1830:2013] -> computing...
  [val] 2m_temperature 2020 shard 12/16 steps [2013:2196] -> computing...
  [val] 2m_temperature 2020 shard 13/16 steps [2196:2379] -> computing...
  [val] 2m_temperature 2020 shard 14/16 steps [2379:2562] ->

test:   0%|          | 0/1 [00:00<?, ?it/s]/var/folders/44/hp8ptt8n3j16gtl4vny36krh0000gn/T/ipykernel_25415/2925846405.py:160: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 32. This could degrade performance. Instead, consider rechunking after loading.
  ds   = xr.open_mfdataset(ps, combine="by_coords",


  [test] 2m_temperature 2021 shard 1/16 steps [0:182] -> computing...
  [test] 2m_temperature 2021 shard 2/16 steps [182:364] -> computing...
  [test] 2m_temperature 2021 shard 3/16 steps [364:546] -> computing...
  [test] 2m_temperature 2021 shard 4/16 steps [546:728] -> computing...
  [test] 2m_temperature 2021 shard 5/16 steps [728:910] -> computing...
  [test] 2m_temperature 2021 shard 6/16 steps [910:1092] -> computing...
  [test] 2m_temperature 2021 shard 7/16 steps [1092:1274] -> computing...
  [test] 2m_temperature 2021 shard 8/16 steps [1274:1456] -> computing...
  [test] 2m_temperature 2021 shard 9/16 steps [1456:1638] -> computing...
  [test] 2m_temperature 2021 shard 10/16 steps [1638:1820] -> computing...
  [test] 2m_temperature 2021 shard 11/16 steps [1820:2002] -> computing...
  [test] 2m_temperature 2021 shard 12/16 steps [2002:2184] -> computing...
  [test] 2m_temperature 2021 shard 13/16 steps [2184:2366] -> computing...
  [test] 2m_temperature 2021 shard 14/16 steps 

test: 100%|██████████| 1/1 [02:02<00:00, 122.57s/it]


Saved lat (133,) lon (133,) -> /Users/thainamhoang/Desktop/ph-reg/data/cerra/44km/2m_temperature/
